# Part 7 — Interoperability and the downstream task

The finished graph exports to every backend it needs from one object: igraph for a
C-speed PageRank written straight back as a layer attribute, PyG `HeteroData` and
CX2 for downstream ML and Cytoscape, and a provenance diff reconstructed from the
snapshots taken along the way. The notebook closes the loop by training a GNN on
the PyG export.

In [ ]:
import pickle
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

import annnet as an
import annnet.utils.plotting  # noqa: F401  (registers an.utils.plotting)
import uc1_common as uc

G = uc.load()
consensus_outputs = pickle.loads(uc.CONSENSUS_PKL.read_bytes())
primary_layer_label = f'consensus_{uc.PRIMARY_COHORT_LABEL}'
G_consensus = G.layers.subgraph_from_layer_tuple((primary_layer_label,))
print({'consensus_layer': primary_layer_label,
       'vertices': G_consensus.global_count('vertices'), 'edges': G_consensus.global_count('edges')})

## Multilayer slicing and consensus plot

`subgraph_from_layer_tuple` returns a real AnnNet subgraph for any layer
coordinate. We slice the consensus layer and render its top-N nodes.

In [ ]:
node_df = consensus_outputs[uc.PRIMARY_COHORT_LABEL]['node_df']
selected = node_df[node_df['selected_for_consensus']].sort_values(
    ['patient_frequency', 'active_count'], ascending=[False, False])
plot_nodes = selected['vertex_id'].head(uc.TOP_N_PLOT).tolist()
G_plot = G_consensus.ops.extract_subgraph(vertices=set(plot_nodes))

fig, ax = plt.subplots(figsize=(12, 12))
try:
    an.utils.plotting.to_matplotlib(G_plot, ax=ax, show_vertex_labels=True, node_size=600)
    ax.set_title(f'Consensus subgraph ({uc.PRIMARY_COHORT_LABEL}, top {len(plot_nodes)} nodes)')
    plt.tight_layout()
    fig.savefig(uc.FIGS / f'consensus_plot_{uc.PRIMARY_COHORT_LABEL}.png', dpi=200, bbox_inches='tight')
    plt.show()
except Exception as exc:
    plt.close(fig)
    print(f'Plot backend issue: {type(exc).__name__}: {exc}')

## Backend swap — igraph PageRank

`G.ig.backend()` converts to igraph once (cached) so PageRank runs in C. igraph
returns scores by vertex index, so we read the matching names off the *same*
backend object and write each score back onto the AnnNet as a layer attribute.

In [ ]:
t0 = time.perf_counter()
ig_sub = G_consensus.ig.backend()          # convert once (cached), then run in C
pageranks = ig_sub.pagerank()
t_pr = time.perf_counter() - t0

# igraph returns scores by vertex index — label them from the same backend object.
pr_map = {uc.bare_vid(n): float(p) for n, p in zip(ig_sub.vs['name'], pageranks)}
consensus_aa = (f'consensus_{uc.PRIMARY_COHORT_LABEL}',)
for vid, p in pr_map.items():
    G.layers.set_vertex_layer_attrs(vid, consensus_aa, pagerank=p)

print(f'igraph pagerank (convert + run): {t_pr * 1000:6.1f} ms')
print('Top consensus nodes by PageRank:')
for vid, p in sorted(pr_map.items(), key=lambda kv: -kv[1])[:8]:
    print(f'  {vid:>10}  {p:.4f}')
G.history.snapshot('after_backend_swap')

## PyG `HeteroData` export

Lossless export of the incidence and attributes into PyG's heterogeneous format,
ready for the GNN below.

In [ ]:
export_rows = []
try:
    pyg_data = an.to_pyg(G, hyperedge_mode='skip')
    torch.save(pyg_data, uc.PYG_OUT)
    export_rows.append({'export': 'PyG HeteroData', 'ok': True, 'path': str(uc.PYG_OUT),
                        'details': f'node_types={list(pyg_data.node_types)}, edge_types={list(pyg_data.edge_types)}'})
except Exception as exc:
    export_rows.append({'export': 'PyG HeteroData', 'ok': False, 'path': str(uc.PYG_OUT),
                        'details': f'{type(exc).__name__}: {exc}'})
export_summary = pd.DataFrame(export_rows)
export_summary.to_csv(uc.TABLES / 'export_summary.csv', index=False)
export_summary

## Provenance — snapshot diff across pipeline stages

Each stage called `G.history.snapshot(label)`; `G.history.diff(a, b)` returns the
structural delta between any two. Together they reconstruct the build after the
fact — an auditable, machine-readable manuscript artifact.

In [ ]:
all_snapshots = list(G.history.list_snapshots())
labels = [s['label'] for s in all_snapshots]
diff_rows = []
for i in range(1, len(labels)):
    d = G.history.diff(labels[i - 1], labels[i])
    diff_rows.append({'from': labels[i - 1], 'to': labels[i],
                      'vertices_added': len(d.vertices_added), 'vertices_removed': len(d.vertices_removed),
                      'edges_added': len(d.edges_added), 'edges_removed': len(d.edges_removed),
                      'slices_added': len(d.slices_added), 'slices_removed': len(d.slices_removed)})
diff_df = pd.DataFrame(diff_rows)
diff_df.to_csv(uc.TABLES / 'history_diff_table.csv', index=False)
pd.DataFrame(all_snapshots)[['label', 'version']].to_csv(uc.TABLES / 'history_snapshots.csv', index=False)
try:
    G.history.export(str(uc.HISTORY_OUT))
except Exception as exc:
    print(f'History export failed: {type(exc).__name__}: {exc}')
print('Structural changes per pipeline stage:')
diff_df

## CX2 export for Cytoscape

One call. The `.cx2` opens directly in Cytoscape for interactive review of the
consensus subnetwork.

In [ ]:
cx2_path = uc.FIGS / f'consensus_{uc.PRIMARY_COHORT_LABEL}.cx2'
an.to_cx2(G_consensus, str(cx2_path))
print(f'Wrote {cx2_path}')

## Save the final AnnNet snapshot

In [ ]:
G.write(str(uc.SNAPSHOT), overwrite=True)
print(f'Wrote final snapshot {uc.SNAPSHOT} ({uc.SNAPSHOT.stat().st_size / 1e6:.2f} MB)')

## Closing the loop — GNN on the PyG export

Train a 2-layer GraphSAGE to predict CancerGeneCensus membership from PKN
topology, against a degree-only logistic baseline. A positive lift confirms the
export carries structural signal beyond raw degree.

In [ ]:
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

pyg_data = torch.load(uc.PYG_OUT, weights_only=False)
node_type = next(iter(pyg_data.node_types))
edge_type = next(iter(pyg_data.edge_types))
n = pyg_data[node_type].num_nodes
ei = pyg_data[edge_type].edge_index
print(f'PyG: {n:,} nodes  {ei.shape[1]:,} edges  ({node_type=}, {edge_type=})')

vattr_df = G.vertex_attributes.to_pandas()
ccg_cols = [c for c in vattr_df.columns if 'CancerGeneCensus' in c]

if not ccg_cols:
    print('No CancerGeneCensus annotation column — GNN demo skipped.')
else:
    pos_vids = set(vattr_df.loc[vattr_df[ccg_cols[0]].notna(), 'vertex_id'])
    idx_to_vid = {idx: vid for vid, idx in pyg_data.manifest['node_index'][node_type].items()}
    y = torch.tensor([1.0 if idx_to_vid[i] in pos_vids else 0.0 for i in range(n)])
    print(f'Labels: {int(y.sum()):,} CancerGeneCensus positive / {n:,} ({y.mean():.1%})')

    deg = torch.zeros(n)
    deg.index_add_(0, ei[0], torch.ones(ei.shape[1]))
    deg.index_add_(0, ei[1], torch.ones(ei.shape[1]))
    x = torch.log1p(deg).unsqueeze(-1)

    torch.manual_seed(uc.SEED)
    perm = torch.randperm(n)
    tr, va = perm[: int(0.7 * n)], perm[int(0.7 * n):]

    class SAGE(torch.nn.Module):
        def __init__(self):
            super().__init__()
            self.c1, self.c2 = SAGEConv(1, 16), SAGEConv(16, 1)

        def forward(self, x, ei):
            return self.c2(F.relu(self.c1(x, ei)), ei)

    model = SAGE()
    opt = torch.optim.Adam(model.parameters(), lr=1e-2, weight_decay=1e-4)
    for _epoch in range(100):
        model.train()
        opt.zero_grad()
        loss = F.binary_cross_entropy_with_logits(model(x, ei).squeeze(-1)[tr], y[tr])
        loss.backward()
        opt.step()

    model.eval()
    with torch.no_grad():
        scores = torch.sigmoid(model(x, ei).squeeze(-1))
    auc_gnn = roc_auc_score(y[va].numpy(), scores[va].numpy())
    lr_clf = LogisticRegression(max_iter=1000).fit(x[tr].numpy(), y[tr].numpy())
    auc_deg = roc_auc_score(y[va].numpy(), lr_clf.predict_proba(x[va].numpy())[:, 1])

    print(f'GraphSAGE  val ROC-AUC: {auc_gnn:.3f}')
    print(f'Degree-only baseline  : {auc_deg:.3f}')
    print(f'Lift over degree      : {auc_gnn - auc_deg:+.3f}')